In [1]:
# Install compatible PyTorch Geometric dependencies

# Install a compatible CPU version of PyTorch
!pip install --force-reinstall torch==2.12.0 torchvision==0.27.0 --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached torch-2.12.0%2Bcpu-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached torchvision-0.27.0%2Bcpu-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached filelock-4.2.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-78.1.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached numpy-2.5.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pillow-13.0.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markups

In [3]:
import torch
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.nn import GCNConv
from torch_geometric.utils import train_test_split_edges

In [4]:
# Install CPU-compatible PyTorch Geometric and its extensions

!pip install -q torch-geometric
!pip install -q pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.12.0+cpu.html

In [5]:
# Diagnose PyTorch and PyG installation

import torch
import sys
import importlib.metadata as metadata

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

for package in ["torchvision", "torch-geometric", "pyg-lib", "torch-scatter", "torch-sparse"]:
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "Not installed")

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
PyTorch: 2.12.0+cpu
CUDA version: None
CUDA available: False
torchvision 0.27.0+cpu
torch-geometric 2.8.1
pyg-lib 0.9.0+pt212cpu
torch-scatter 2.1.2+pt212cpu
torch-sparse 0.6.18+pt212cpu


# Tutorial 6  
Graph AutoEncoders GAE &  
Variational Graph Autoencoders VGAE    

[paper](https://arxiv.org/pdf/1611.07308.pdf)  
[code](https://github.com/rusty1s/pytorch_geometric/blob/master/examples/autoencoder.py)

## Graph AutoEncoder GAE

### Load the data

In [6]:
dataset = Planetoid("\..", "CiteSeer", transform=T.NormalizeFeatures())
dataset.data

<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:1: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_8194/2929005015.py:1: SyntaxWarning: invalid escape sequence '\.'
  dataset = Planetoid("\..", "CiteSeer", transform=T.NormalizeFeatures())
Processing...
Done!
/tmp/ipykernel_8194/2929005015.py:2: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  dataset.data


Data(x=[3327, 3703], edge_index=[2, 9104], y=[3327], train_mask=[3327], val_mask=[3327], test_mask=[3327])

In [7]:
data = dataset[0]
data.train_mask = data.val_mask = data.test_mask = None
data

Data(x=[3327, 3703], edge_index=[2, 9104], y=[3327])

In [8]:
data = train_test_split_edges(data)

/tmp/ipykernel_8194/3076777706.py:1: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  data = train_test_split_edges(data)


In [9]:
data

Data(x=[3327, 3703], y=[3327], val_pos_edge_index=[2, 227], test_pos_edge_index=[2, 455], train_pos_edge_index=[2, 7740], train_neg_adj_mask=[3327, 3327], val_neg_edge_index=[2, 227], test_neg_edge_index=[2, 455])

### Define the Encoder

In [10]:
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GCNEncoder, self).__init__()
        self.conv1 = GCNConv(in_channels, 2 * out_channels, cached=True) # cached only for transductive learning
        self.conv2 = GCNConv(2 * out_channels, out_channels, cached=True) # cached only for transductive learning

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)


### Define the Autoencoder

In [11]:
from torch_geometric.nn import GAE

In [12]:
# parameters
out_channels = 2
num_features = dataset.num_features
epochs = 100

# model
model = GAE(GCNEncoder(num_features, out_channels))

# move to GPU (if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
x = data.x.to(device)
train_pos_edge_index = data.train_pos_edge_index.to(device)

# inizialize the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [13]:
def train():
    model.train()
    optimizer.zero_grad()
    z = model.encode(x, train_pos_edge_index)
    loss = model.recon_loss(z, train_pos_edge_index)
    #if args.variational:
    #   loss = loss + (1 / data.num_nodes) * model.kl_loss()
    loss.backward()
    optimizer.step()
    return float(loss)


def test(pos_edge_index, neg_edge_index):
    model.eval()
    with torch.no_grad():
        z = model.encode(x, train_pos_edge_index)
    return model.test(z, pos_edge_index, neg_edge_index)



In [14]:
for epoch in range(1, epochs + 1):
    loss = train()

    auc, ap = test(data.test_pos_edge_index, data.test_neg_edge_index)
    print('Epoch: {:03d}, AUC: {:.4f}, AP: {:.4f}'.format(epoch, auc, ap))

/tmp/ipykernel_8194/2919475705.py:10: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:838.)
  return float(loss)


Epoch: 001, AUC: 0.6336, AP: 0.6804
Epoch: 002, AUC: 0.6574, AP: 0.7047
Epoch: 003, AUC: 0.6710, AP: 0.7099
Epoch: 004, AUC: 0.6753, AP: 0.7143
Epoch: 005, AUC: 0.6778, AP: 0.7177
Epoch: 006, AUC: 0.6793, AP: 0.7220
Epoch: 007, AUC: 0.6815, AP: 0.7268
Epoch: 008, AUC: 0.6837, AP: 0.7294
Epoch: 009, AUC: 0.6863, AP: 0.7313
Epoch: 010, AUC: 0.6881, AP: 0.7329
Epoch: 011, AUC: 0.6892, AP: 0.7341
Epoch: 012, AUC: 0.6900, AP: 0.7356
Epoch: 013, AUC: 0.6898, AP: 0.7374
Epoch: 014, AUC: 0.6896, AP: 0.7393
Epoch: 015, AUC: 0.6880, AP: 0.7402
Epoch: 016, AUC: 0.6864, AP: 0.7407
Epoch: 017, AUC: 0.6851, AP: 0.7419
Epoch: 018, AUC: 0.6839, AP: 0.7425
Epoch: 019, AUC: 0.6830, AP: 0.7427
Epoch: 020, AUC: 0.6826, AP: 0.7430
Epoch: 021, AUC: 0.6825, AP: 0.7434
Epoch: 022, AUC: 0.6827, AP: 0.7434
Epoch: 023, AUC: 0.6831, AP: 0.7438
Epoch: 024, AUC: 0.6834, AP: 0.7439
Epoch: 025, AUC: 0.6840, AP: 0.7442
Epoch: 026, AUC: 0.6846, AP: 0.7445
Epoch: 027, AUC: 0.6856, AP: 0.7448
Epoch: 028, AUC: 0.6872, AP:

In [15]:
Z = model.encode(x, train_pos_edge_index)
Z

tensor([[-0.3065,  0.4722],
        [ 0.8718, -0.9890],
        [-0.7305,  0.8105],
        ...,
        [ 0.4983, -0.4476],
        [-0.6362,  0.7426],
        [-0.8381,  0.8727]], grad_fn=<AddBackward0>)

## Are the results (AUC) and (AP) easy to read and compare?

# Use Tensorboard

In [16]:
from torch.utils.tensorboard import SummaryWriter

In [17]:
# parameters
out_channels = 2
num_features = dataset.num_features
epochs = 100

# model
model = GAE(GCNEncoder(num_features, out_channels))

# move to GPU (if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
x = data.x.to(device)
train_pos_edge_index = data.train_pos_edge_index.to(device)

# inizialize the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

### Import tensorboard

#### Installation: (if needed) "pip install tensorboard"

In [18]:
writer = SummaryWriter('runs/GAE1_experiment_'+'2d_100_epochs')

In [19]:
for epoch in range(1, epochs + 1):
    loss = train()
    auc, ap = test(data.test_pos_edge_index, data.test_neg_edge_index)
    print('Epoch: {:03d}, AUC: {:.4f}, AP: {:.4f}'.format(epoch, auc, ap))


    writer.add_scalar('auc train',auc,epoch) # new line
    writer.add_scalar('ap train',ap,epoch)   # new line

Epoch: 001, AUC: 0.6794, AP: 0.6960
Epoch: 002, AUC: 0.6760, AP: 0.6929
Epoch: 003, AUC: 0.6571, AP: 0.6880
Epoch: 004, AUC: 0.6627, AP: 0.6966
Epoch: 005, AUC: 0.6711, AP: 0.7049
Epoch: 006, AUC: 0.6789, AP: 0.7115
Epoch: 007, AUC: 0.6852, AP: 0.7171
Epoch: 008, AUC: 0.6881, AP: 0.7196
Epoch: 009, AUC: 0.6905, AP: 0.7222
Epoch: 010, AUC: 0.6923, AP: 0.7242
Epoch: 011, AUC: 0.6943, AP: 0.7259
Epoch: 012, AUC: 0.6962, AP: 0.7283
Epoch: 013, AUC: 0.6979, AP: 0.7303
Epoch: 014, AUC: 0.6992, AP: 0.7326
Epoch: 015, AUC: 0.7005, AP: 0.7349
Epoch: 016, AUC: 0.7013, AP: 0.7368
Epoch: 017, AUC: 0.7016, AP: 0.7387
Epoch: 018, AUC: 0.7012, AP: 0.7405
Epoch: 019, AUC: 0.7009, AP: 0.7424
Epoch: 020, AUC: 0.6994, AP: 0.7432
Epoch: 021, AUC: 0.6981, AP: 0.7437
Epoch: 022, AUC: 0.6967, AP: 0.7444
Epoch: 023, AUC: 0.6957, AP: 0.7447
Epoch: 024, AUC: 0.6951, AP: 0.7451
Epoch: 025, AUC: 0.6949, AP: 0.7455
Epoch: 026, AUC: 0.6947, AP: 0.7459
Epoch: 027, AUC: 0.6942, AP: 0.7457
Epoch: 028, AUC: 0.6940, AP:

## Graph Variational AutoEncoder (GVAE)

In [20]:
from torch_geometric.nn import VGAE

In [21]:
dataset = Planetoid("\..", "CiteSeer", transform=T.NormalizeFeatures())
data = dataset[0]
data.train_mask = data.val_mask = data.test_mask = data.y = None
data = train_test_split_edges(data)


class VariationalGCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super(VariationalGCNEncoder, self).__init__()
        self.conv1 = GCNConv(in_channels, 2 * out_channels, cached=True) # cached only for transductive learning
        self.conv_mu = GCNConv(2 * out_channels, out_channels, cached=True)
        self.conv_logstd = GCNConv(2 * out_channels, out_channels, cached=True)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:1: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_8194/3226588792.py:1: SyntaxWarning: invalid escape sequence '\.'
  dataset = Planetoid("\..", "CiteSeer", transform=T.NormalizeFeatures())
/tmp/ipykernel_8194/3226588792.py:4: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  data = train_test_split_edges(data)


In [22]:
out_channels = 2
num_features = dataset.num_features
epochs = 300


model = VGAE(VariationalGCNEncoder(num_features, out_channels))  # new line

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
x = data.x.to(device)
train_pos_edge_index = data.train_pos_edge_index.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [23]:
def train():
    model.train()
    optimizer.zero_grad()
    z = model.encode(x, train_pos_edge_index)
    loss = model.recon_loss(z, train_pos_edge_index)

    loss = loss + (1 / data.num_nodes) * model.kl_loss()  # new line
    loss.backward()
    optimizer.step()
    return float(loss)


def test(pos_edge_index, neg_edge_index):
    model.eval()
    with torch.no_grad():
        z = model.encode(x, train_pos_edge_index)
    return model.test(z, pos_edge_index, neg_edge_index)

In [24]:
writer = SummaryWriter('runs/VGAE_experiment_'+'2d_100_epochs')

for epoch in range(1, epochs + 1):
    loss = train()
    auc, ap = test(data.test_pos_edge_index, data.test_neg_edge_index)
    print('Epoch: {:03d}, AUC: {:.4f}, AP: {:.4f}'.format(epoch, auc, ap))


    writer.add_scalar('auc train',auc,epoch) # new line
    writer.add_scalar('ap train',ap,epoch)   # new line

Epoch: 001, AUC: 0.5967, AP: 0.6126
Epoch: 002, AUC: 0.6288, AP: 0.6538
Epoch: 003, AUC: 0.6416, AP: 0.6668
Epoch: 004, AUC: 0.6451, AP: 0.6699
Epoch: 005, AUC: 0.6456, AP: 0.6703
Epoch: 006, AUC: 0.6451, AP: 0.6697
Epoch: 007, AUC: 0.6451, AP: 0.6692
Epoch: 008, AUC: 0.6449, AP: 0.6689
Epoch: 009, AUC: 0.6446, AP: 0.6685
Epoch: 010, AUC: 0.6444, AP: 0.6677
Epoch: 011, AUC: 0.6443, AP: 0.6667
Epoch: 012, AUC: 0.6446, AP: 0.6663
Epoch: 013, AUC: 0.6445, AP: 0.6658
Epoch: 014, AUC: 0.6445, AP: 0.6651
Epoch: 015, AUC: 0.6445, AP: 0.6646
Epoch: 016, AUC: 0.6444, AP: 0.6644
Epoch: 017, AUC: 0.6444, AP: 0.6643
Epoch: 018, AUC: 0.6448, AP: 0.6645
Epoch: 019, AUC: 0.6452, AP: 0.6650
Epoch: 020, AUC: 0.6456, AP: 0.6656
Epoch: 021, AUC: 0.6461, AP: 0.6662
Epoch: 022, AUC: 0.6465, AP: 0.6670
Epoch: 023, AUC: 0.6474, AP: 0.6686
Epoch: 024, AUC: 0.6480, AP: 0.6697
Epoch: 025, AUC: 0.6485, AP: 0.6714
Epoch: 026, AUC: 0.6493, AP: 0.6730
Epoch: 027, AUC: 0.6499, AP: 0.6743
Epoch: 028, AUC: 0.6506, AP: